In [ ]:
import os
import dotenv
import geopandas as gpd
import pandas as pd
import numpy as np
import logging
from datetime import datetime

gpd.options.io_engine = "pyogrio"
os.environ["PYOGRIO_USE_ARROW"] = "1"

dotenv.load_dotenv()
output_parquets=os.getenv('OUTPUT_PARQUETS')
input_parquet=os.getenv("INPUT_PARQUETS")
ret_class=os.path.join(input_parquet,'rec_class_all.parquet')
wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
aflb_parquet=os.getenv("ALFB_PARQUET")
cia_dissolve=os.path.join(input_parquet,"cia_dissolve.parquet")
step2d_base=os.path.join(output_parquets,"step_2d_base_aflb.parquet")
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

#variables 
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
input_data_gdb=os.path.join(source_data,'Scen_5_inputs.gdb')
cia_ciap_l='cultural_areas'
hv1_l='hv1_clip'
tsu_l='TSU'

import duckdb
conn = duckdb.connect()
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.install_extension("arrow")
conn.load_extension("spatial")
conn.load_extension("httpfs")
conn.load_extension("arrow")

In [ ]:
rec=gpd.read_parquet(ret_class)
rec.to_parquet(ret_class)
rep_map = {
    'High Productivity Coniferous Leading': 'HPC',
    'High Productivity Deciduous Leading': 'HPD',
    'High Productivity Mixedwood': 'HPM',
    'Medium Productivity Coniferous Leading': 'MPC',
    'Medium Productivity Deciduous Leading': 'MPD',
    'Medium Productivity Mixedwood': 'MPM',
    'Low Productivity Coniferous Leading': 'LPC',
    'Low Productivity Deciduous Leading': 'LPD',
    'Low Productivity Mixedwood': 'LPM',
}
rec['Rec_Cat_short'] = rec['FOR_REP'].map(rep_map)

In [ ]:

for_rep_area=rec.groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()
for_rep_area['desired total(25%) area(ha)']=for_rep_area['total_aflb_ha']*0.25

rec['aflb_area_ha']=rec['aflb_fact']*(rec.geometry.area/10000)
rec_cat_for_rep=rec.groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short']).agg(total_aflb_ha=('aflb_area_ha', 'sum')).reset_index()

In [ ]:
def creat_rec_tables(wmb_name:str, rec_cat_for_rep:pd.DataFrame) -> pd.DataFrame:
    """
    Create recruitment tables for a given water management basin.

    Parameters:
    wmb_name (str): The name of the water management basin.
    rec_cat_for_rep (pd.DataFrame): DataFrame containing recruitment categories and their corresponding areas.

    Returns:
    pd.DataFrame: A DataFrame containing the recruitment table for the specified water management basin.
    """
    wmb = (
        rec_cat_for_rep[
            (rec_cat_for_rep['WATER_MANAGEMENT_BASIN_NAME'] == wmb_name) &
            (rec_cat_for_rep['Rec_Cat'].isin([1, 2, 3, 4, 5]))
        ]
        .drop(columns='WATER_MANAGEMENT_BASIN_NAME')
    )

    pivot = (
        wmb
        .pivot_table(
            index='Rec_Cat',
            columns='Rec_Cat_short',
            aggfunc='sum',
            fill_value=0
        )
    )
    pivot.index = pivot.index.astype(str)
    pivot.loc['25% total aflb'] = pivot.sum() * 0.25

    target = pivot.loc['25% total aflb']
    data = pivot.drop(index='25% total aflb')
    data = data.sort_index(key=lambda x: x.astype(float))
    csum = data.cumsum()
    capped_csum = csum.clip(upper=target, axis=1)
    result = capped_csum.diff().fillna(capped_csum)
    final = pd.concat([result, target.to_frame().T])
    final.drop(index='25% total aflb').sum().round(6) == target.round(6)
    data = data.sort_index(key=lambda x: x.astype(float))
    return final.reset_index()

In [ ]:
bbr=creat_rec_tables('Blueberry River', rec_cat_for_rep)
lsc=creat_rec_tables('Lower Sikanni Chief River', rec_cat_for_rep)
mbr=creat_rec_tables('Middle Beatton River', rec_cat_for_rep)
ubr=creat_rec_tables('Upper Beatton River', rec_cat_for_rep)


In [ ]:
rec_output_path=os.path.join(table_outputs, f"recruitment_tables_{datetime.now().strftime('%Y-%m-%d')}.xlsx")
with pd.ExcelWriter(rec_output_path, engine="openpyxl") as writer:
    startrow = 0

    for name, df in [
        ("Blueberry River", bbr),
        ("Lower Sikanni Chief River", lsc),
        ("Middle Beatton River", mbr),
        ("Upper Beatton River", ubr),
    ]:
        # Write a title row
        pd.DataFrame([[name]]).to_excel(
            writer,
            sheet_name="Recruitment Tables",
            startrow=startrow,
            index=False,
            header=False
        )

        # Write the table just below the title
        df.to_excel(
            writer,
            sheet_name="Recruitment Tables",
            startrow=startrow + 1,
            index=True
        )

        # Advance startrow (table height + spacing)
        startrow += len(df) + 4
